# Scalar Chebyshev2 vs Trapezoid: Random Degree-4 Polynomials

This notebook uses three fixed random degree-4 polynomials. Each polynomial is defined by random values in `[-0.5, 0.5]` at five CGL nodes on `[0, 1]`, then evaluated and integrated with GTSAM `Chebyshev2` machinery. Each subplot fixes `m`, the number of samples; the y-axis is `N`, the Chebyshev2 CGL-node count (`n=N-1`). The light grey dashed line marks `sqrt(m)`.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import imuFactors.spectral as spectral
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / "python").exists():
    repo_root = repo_root.parent
python_dir = repo_root / "python"
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

import imuFactors.scalar_quadrature as scalar_quadrature

plt.rcParams.update({"figure.dpi": 120})

In [ ]:
INTERVAL = (0.0, 1.0)
POLYNOMIAL_N = 5
POLYNOMIAL_SEED = 8675309

rng = np.random.default_rng(POLYNOMIAL_SEED)
polynomial_node_times = spectral.chebyshev2_points(POLYNOMIAL_N, INTERVAL)
polynomial_node_values = rng.uniform(
    -0.5, 0.5, size=(3, POLYNOMIAL_N)
)
FUNCTIONS = [
    scalar_quadrature.scalar_function_from_chebyshev2_nodes(
        f"random degree-4 polynomial {index + 1}", node_values, INTERVAL
    )
    for index, node_values in enumerate(polynomial_node_values)
]

pd.DataFrame(
    polynomial_node_values,
    columns=[f"f({time:.3f})" for time in polynomial_node_times],
    index=[function.name for function in FUNCTIONS],
)

In [ ]:
M_VALUES = [10, 20, 30, 40, 50]
N_VALUES = np.arange(2, 11)
NOISE_FRACTIONS = np.array([
    0.0, 0.025, 0.05, 0.06, 0.075, 0.10,
    0.12, 0.15, 0.17, 0.20, 0.225,
])
NUM_SEEDS = 100
RANDOM_SEED = 20260523
EVALUATION_COUNT = 151

N_values_by_m = {
    m: N_VALUES
    for m in M_VALUES
}
pd.DataFrame(
    {
        "m": list(N_values_by_m),
        "N_min": [values[0] for values in N_values_by_m.values()],
        "N_max": [values[-1] for values in N_values_by_m.values()],
        "sqrt_m": [np.sqrt(m) for m in N_values_by_m],
        "num_N": [len(values) for values in N_values_by_m.values()],
    }
)


In [ ]:
runs = []
for m, N_grid in N_values_by_m.items():
    runs.append(
        scalar_quadrature.run_scalar_monte_carlo(
            FUNCTIONS,
            m_values=[m],
            N_values=N_grid,
            noise_fractions=NOISE_FRACTIONS,
            num_seeds=NUM_SEEDS,
            seed=RANDOM_SEED,
            interval=INTERVAL,
            evaluation_count=EVALUATION_COUNT,
        )
    )

method_metrics = pd.concat(
    [run.method_metrics for run in runs], ignore_index=True
)
comparisons = pd.concat(
    [run.comparisons for run in runs], ignore_index=True
)
comparisons.head()


In [ ]:
for function in FUNCTIONS:
    fig = scalar_quadrature.plot_fixed_m_comparison(
        comparisons,
        function_name=function.name,
        selected_m_values=M_VALUES,
        show_sqrt_m=True,
    )
    display(fig)
    plt.close(fig)

In [ ]:
summary = (
    comparisons.groupby(["function", "m"])[["end_error", "rmse_error", "max_error"]]
    .median()
    .round(6)
)
summary

## Decision-oriented Plotly views

These views use the same comparison dataframe as the heatmaps above. Positive advantage is `trapezoid error - Chebyshev2 error`, so values above zero favor Chebyshev2. The diamond marks the best median RMSE `N`; the light grey dashed line is `sqrt(m)`. The table reports ideal `N` by metric plus a rank-based robust `N`.


In [ ]:
for function in FUNCTIONS:
    display(
        scalar_quadrature.plot_advantage_curves_by_m(
            comparisons,
            function.name,
            selected_m_values=M_VALUES,
            metric="rmse_error",
            y_range_min_N=POLYNOMIAL_N,
        )
    )
    display(
        scalar_quadrature.plot_robust_N_table(
            comparisons,
            function.name,
            selected_m_values=M_VALUES,
        )
    )